In [1]:
import numpy as np; import pandas as pd; import matplotlib.pyplot as plt
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, Callback
from tensorflow.keras.optimizers import Adam
import time
import warnings; warnings.filterwarnings('ignore')

In [2]:
class EpochTimer(Callback):
    def on_train_begin(self, logs=None): self.times = []
    def on_epoch_begin(self, epoch, logs=None): self._start = time.time()
    def on_epoch_end(self, epoch, logs=None): self.times.append(time.time() - self._start)

# Task 1
### Data Preparation and Preprocessing
- Load the fashion-MNIST dataset (training and testing) using the `load_data()` method from the `fmnist` package
- Convert the cloth items class labels into one-hot encoded format with 10 output classes
- Rescale inputs to the range $[0, 1]$
- Display the shape of the training and testing datasets

In [3]:
(x_train, y_train), (x_val, y_val) = fashion_mnist.load_data()
y_train_cat = to_categorical(y_train, 10); y_val_cat = to_categorical(y_val, 10)
x_train = x_train / 255; x_val = x_val / 255
print('Training shape:', x_train.shape); print('Test shape:', x_val.shape)

Training shape: (60000, 28, 28)
Test shape: (10000, 28, 28)


# Task 2
### FCFNN
- Reshape each image from 28 × 28 into a 1D vector of length 784
- Build an FCFNN using the `Sequential()` API from `keras` using a suitable combination of the following layers
  - `Dense()` layers with suitable number of  neurons
  - `Dropout()` layers with a suitable rates
  - `Dense()` output layer with 10 neurons and `'softmax'` activation
  - Begin with ReLU activation for all layers except the output layer
- Study the model architecture using the `summary()` method
- Compile the model using `Adam()` optimizer, `'categorical_crossentropy'` loss, `'accuracy'` as evaluation metric
- Train the model for suitable number of epochs with a suitable batch size, and provide the validation data separately during training
- Limit training of the model using `EarlyStopping()` from `keras` with a suitable tolerance
- Feel free to change the architecture of the model (number of layers, number of neurons, activation functions, batch size, number of epochs, early stopping specifics, optimizer learning rate, and so on) and try to improve the model
- Note down the final specifics of the model such as parameter count, average training time per epoch, and performance

In [4]:
x_train_fcfnn = x_train.reshape(-1, 784); x_val_fcfnn = x_val.reshape(-1, 784)
model_fcfnn = Sequential([Dense(units = 32, activation = 'relu', input_shape = (784,), name = 'dense_1'),
                          Dropout(rate = 0.05, name = 'dropout_1'),
                          Dense(units = 64, activation = 'relu', name = 'dense_2'),
                          Dropout(rate = 0.05, name = 'dropout_2'),
                          Dense(units = 10, activation = 'softmax', name = 'output')],
                         name = 'fcfnn')
model_fcfnn.summary()
fcfnn_params = model_fcfnn.count_params()

Model: "fcfnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_1 (Dense)                      │ (None, 32)                  │          25,120 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 64)                  │           2,112 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ output (Dense)                       │ (None, 10)                  │             650 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 27,882 (108.91 KB)

 Trainable params: 27,882 (108.91 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
early_stop = EarlyStopping(monitor = 'val_loss', patience = 3, restore_best_weights = True)

In [6]:
model_fcfnn.compile(optimizer = Adam(learning_rate = 0.001), loss = 'categorical_crossentropy', metrics = ['accuracy'])
timer = EpochTimer()
history_fcfnn = model_fcfnn.fit(x_train_fcfnn, y_train_cat, validation_split = 0.2,
                                epochs = 5, batch_size = 32, callbacks = [early_stop, timer])
avg_time_fcfnn = sum(timer.times) / len(timer.times)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.7825 - loss: 0.6131 - val_accuracy: 0.8373 - val_loss: 0.4524
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8446 - loss: 0.4309 - val_accuracy: 0.8588 - val_loss: 0.3911
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8551 - loss: 0.3937 - val_accuracy: 0.8652 - val_loss: 0.3809
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.8654 - loss: 0.3719 - val_accuracy: 0.8676 - val_loss: 0.3606
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.8697 - loss: 0.3544 - val_accuracy: 0.8652 - val_loss: 0.3792


In [7]:
fcfnn_train_loss, fcfnn_train_acc = model_fcfnn.evaluate(x_train_fcfnn, y_train_cat)
fcfnn_val_loss, fcfnn_val_acc = model_fcfnn.evaluate(x_val_fcfnn, y_val_cat)

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.8791 - loss: 0.3286
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8631 - loss: 0.3806


In [8]:
hist_df = pd.DataFrame(history_fcfnn.history); hist_df.insert(0, 'epoch', np.arange(1, len(hist_df) + 1))
hist_df.set_index('epoch', inplace = True); hist_df

,accuracy,loss,val_accuracy,val_loss
epoch,,,,
1,0.782479,0.613141,0.837333,0.452430
2,0.844562,0.430909,0.858833,0.391122
3,0.855125,0.393717,0.865167,0.380911
4,0.865396,0.371887,0.867583,0.360634
5,0.869687,0.354446,0.865167,0.379160
